# MLP Activation Profiling

Profile MLP hidden activations: pre/post nonlinearity statistics,
neuron distributions, position profiles, and pre-post correlations.

## Why This Matters

MLP activation provides tools for understanding how feed-forward layers transform and store information. Since MLPs have been shown to function as key-value memories, understanding their internal mechanics is essential for locating knowledge and understanding factual recall.

**Key references:**
- [Geva et al. (2021) "Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913) — Shows MLPs function as key-value memory stores
- [Bills et al. (2023) "Language Models Can Explain Neurons"](https://openaipublic.blob.core.windows.net/neuron-explainer/paper/index.html) — Automated neuron interpretation methods

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.mlp_activation_profiling import (
    mlp_pre_activation_profile, mlp_post_activation_profile,
    mlp_neuron_activation_distribution, mlp_activation_position_profile,
    mlp_pre_post_correlation,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Pre-Activation Profile

Magnitude and distribution of MLP inputs (before nonlinearity).

In [ ]:
result = mlp_pre_activation_profile(model, tokens)
for p in result['per_layer']:
    bar = '█' * int(p['fraction_positive'] * 20)
    print(f"  Layer {p['layer']}: norm={p['mean_norm']:.4f}±{p['std_norm']:.4f}, "
          f"max={p['max_activation']:.4f}, pos_frac={p['fraction_positive']:.1%} {bar}")

## Post-Activation Profile

Magnitude and sparsity after the activation function.

In [ ]:
result = mlp_post_activation_profile(model, tokens)
for p in result['per_layer']:
    bar = '░' * int(p['sparsity'] * 20) + '█' * int((1 - p['sparsity']) * 20)
    print(f"  Layer {p['layer']}: norm={p['mean_norm']:.4f}, "
          f"sparsity={p['sparsity']:.1%}, max={p['max_activation']:.4f} {bar}")

## Neuron Activation Distribution

Most active and dead neurons at each layer.

In [ ]:
for layer in range(4):
    result = mlp_neuron_activation_distribution(model, tokens, layer=layer, top_k=5)
    print(f"Layer {layer}: {result['n_dead']}/{result['d_mlp']} dead ({result['dead_fraction']:.1%})")
    for n in result['top_neurons'][:3]:
        print(f"  Neuron {n['neuron']}: mean={n['mean_activation']:.4f}, max={n['max_activation']:.4f}")
    print()

## Position Profile

How do MLP activations vary across input positions?

In [ ]:
result = mlp_activation_position_profile(model, tokens, layer=0)
for p in result['per_position']:
    bar = '█' * int(p['n_active_neurons'] / 10)
    print(f"  Pos {p['position']} (tok {p['token']:2d}): "
          f"norm={p['activation_norm']:.4f}, "
          f"sparsity={p['sparsity']:.1%}, "
          f"active={p['n_active_neurons']} {bar}")

## Pre-Post Correlation

How does the nonlinearity transform the signal?

In [ ]:
result = mlp_pre_post_correlation(model, tokens)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: corr={p['norm_correlation']:.4f}, "
          f"compression={p['compression_ratio']:.4f}, "
          f"pre={p['pre_mean_norm']:.4f}→post={p['post_mean_norm']:.4f}")